# 04 — PyTorch Baseline: 1D-CNN for Hydraulic Fault Diagnosis

**목표**: UCI Hydraulic Systems 17개 센서 시계열 → 1D-CNN 단일 타깃 분류 베이스라인

**구성**:
1. 데이터 로드 & 전처리 (multi-rate → 공통 길이 정렬)
2. PyTorch Dataset / DataLoader
3. 1D-CNN 모델 정의
4. 학습 루프 & 평가
5. 4개 타깃 각각 독립 학습 → 결과 비교

**타깃**: cooler (3 class) · valve (4 class) · pump (3 class) · accumulator (4 class)

## 0. 환경 설정 (Colab)

In [ ]:
import sys, os

# Colab: 프로젝트 루트 설정
if "google.colab" in sys.modules:
    REPO = "/content/hydraulic-phm-internship"
    if not os.path.exists(REPO):
        !git clone https://github.com/pjtae1026-blip/hydraulic-phm-internship.git
    os.chdir(REPO)
    !pip install -q pywavelets scikit-fda umap-learn
    # 데이터 다운로드
    if not os.path.exists("data/uci/PS1.txt"):
        !python scripts/download_data.py

sys.path.insert(0, ".")
print(f"Working dir: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 1. 데이터 로드 & 전처리

17개 센서는 샘플링 레이트가 다릅니다 (100Hz/10Hz/1Hz → 6000/600/60 timesteps).  
CNN 입력을 위해 모든 센서를 **60 timesteps**로 다운샘플링하여 `(2205, 17, 60)` 텐서를 구성합니다.

In [ ]:
from src.data_loader import load_all_sensors, load_labels, SENSOR_SPECS
from src.preprocessing import normalize_zscore

data_dir = __import__("pathlib").Path("data/uci")
sensors = load_all_sensors(data_dir)
labels = load_labels(data_dir)

print(f"\nCycles: {labels.shape[0]}")
print(f"Label distribution:")
for col in ["cooler", "valve", "pump", "accumulator"]:
    print(f"  {col}: {dict(labels[col].value_counts().sort_index())}")

In [ ]:
TARGET_LEN = 60

def build_tensor(sensors, target_len=TARGET_LEN):
    """모든 센서를 target_len으로 맞추고 (n_cycles, n_sensors, target_len) 텐서 생성."""
    arrays = []
    for name in SENSOR_SPECS:
        X = sensors[name]  # (2205, cols)
        n_cycles, n_cols = X.shape

        # 다운샘플링: 균등 간격 인덱스 선택
        if n_cols > target_len:
            idx = np.linspace(0, n_cols - 1, target_len, dtype=int)
            X = X[:, idx]

        # per-sensor z-score 정규화 (전체 사이클 기준)
        mu = X.mean()
        std = X.std()
        if std > 0:
            X = (X - mu) / std

        arrays.append(X)

    # (n_cycles, 17, target_len)
    return np.stack(arrays, axis=1)

X_all = build_tensor(sensors)
print(f"Input tensor shape: {X_all.shape}")  # (2205, 17, 60)
print(f"Memory: {X_all.nbytes / 1e6:.1f} MB")

## 2. PyTorch Dataset & DataLoader

In [ ]:
class HydraulicDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def prepare_data(X_all, labels, target_col, test_size=0.2, batch_size=64):
    """타깃 라벨을 0-indexed로 변환하고 train/test split + DataLoader 생성."""
    le = LabelEncoder()
    y = le.fit_transform(labels[target_col].values)
    n_classes = len(le.classes_)

    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y, test_size=test_size, random_state=SEED, stratify=y
    )

    train_ds = HydraulicDataset(X_train, y_train)
    test_ds = HydraulicDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader, le, n_classes, X_test, y_test

## 3. 1D-CNN 모델

구조: `Conv1d → BN → ReLU → MaxPool` × 3 블록 → Global Average Pooling → FC

- 입력: `(batch, 17, 60)` — 17개 센서를 채널로 취급
- Conv 필터: 32 → 64 → 128
- Kernel size: 5

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, n_channels=17, seq_len=60, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv1d(n_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            # Block 2
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            # Block 3
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
        )

        self.gap = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)

# 모델 구조 확인
model_test = CNN1D(n_channels=17, seq_len=60, n_classes=4).to(device)
dummy = torch.randn(2, 17, 60).to(device)
out = model_test(dummy)
print(f"Input:  {dummy.shape}")
print(f"Output: {out.shape}")
print(f"Parameters: {sum(p.numel() for p in model_test.parameters()):,}")
del model_test, dummy, out

## 4. 학습 & 평가 함수

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y_batch)
        correct += (logits.argmax(1) == y_batch).sum().item()
        total += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        total_loss += loss.item() * len(y_batch)
        preds = logits.argmax(1)
        correct += (preds == y_batch).sum().item()
        total += len(y_batch)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())
    return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def train_model(target_col, n_epochs=50, lr=1e-3, batch_size=64):
    """특정 타깃에 대해 1D-CNN 학습 후 결과 반환."""
    print(f"\n{'='*60}")
    print(f"  Target: {target_col}")
    print(f"{'='*60}")

    train_loader, test_loader, le, n_classes, X_test, y_test = \
        prepare_data(X_all, labels, target_col, batch_size=batch_size)

    model = CNN1D(n_channels=17, seq_len=TARGET_LEN, n_classes=n_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0
    best_state = None

    for epoch in range(n_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, test_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    # best 모델로 최종 평가
    model.load_state_dict(best_state)
    model.to(device)
    _, final_acc, preds, true_labels = evaluate(model, test_loader, criterion)

    print(f"\n  Best Val Accuracy: {best_val_acc:.4f}")
    print(f"\n  Classification Report:")
    target_names = [str(c) for c in le.classes_]
    print(classification_report(true_labels, preds, target_names=target_names))

    return model, history, le, preds, true_labels

## 5. 4개 타깃 학습 실행

cooler, valve, pump, accumulator 각각에 대해 독립적으로 1D-CNN을 학습합니다.

In [ ]:
TARGETS = ["cooler", "valve", "pump", "accumulator"]
results = {}

for target in TARGETS:
    model, history, le, preds, true_labels = train_model(target, n_epochs=50, lr=1e-3)
    results[target] = {
        "model": model,
        "history": history,
        "label_encoder": le,
        "preds": preds,
        "true_labels": true_labels,
    }

## 6. 결과 시각화

### 6.1 학습 곡선 (Loss & Accuracy)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for i, target in enumerate(TARGETS):
    h = results[target]["history"]
    epochs = range(1, len(h["train_loss"]) + 1)

    # Loss
    axes[0, i].plot(epochs, h["train_loss"], label="Train")
    axes[0, i].plot(epochs, h["val_loss"], label="Val")
    axes[0, i].set_title(f"{target} — Loss")
    axes[0, i].set_xlabel("Epoch")
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)

    # Accuracy
    axes[1, i].plot(epochs, h["train_acc"], label="Train")
    axes[1, i].plot(epochs, h["val_acc"], label="Val")
    axes[1, i].set_title(f"{target} — Accuracy")
    axes[1, i].set_xlabel("Epoch")
    axes[1, i].set_ylim(0, 1.05)
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("reports/04_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for i, target in enumerate(TARGETS):
    r = results[target]
    cm = confusion_matrix(r["true_labels"], r["preds"])
    class_names = [str(c) for c in r["label_encoder"].classes_]

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[i])
    axes[i].set_title(f"{target}")
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True")

plt.suptitle("Confusion Matrices — 1D-CNN Baseline", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("reports/04_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.3 결과 요약 테이블

In [ ]:
summary_rows = []
for target in TARGETS:
    r = results[target]
    best_val_acc = max(r["history"]["val_acc"])
    test_acc = accuracy_score(r["true_labels"], r["preds"])
    n_classes = len(r["label_encoder"].classes_)
    summary_rows.append({
        "Target": target,
        "Classes": n_classes,
        "Best Val Acc": f"{best_val_acc:.4f}",
        "Test Acc": f"{test_acc:.4f}",
    })

summary_df = pd.DataFrame(summary_rows)
print("=" * 50)
print("  1D-CNN Baseline Results Summary")
print("=" * 50)
print(summary_df.to_string(index=False))
print("=" * 50)

## 7. 분석 & Next Steps

### 관찰
- 1D-CNN은 17개 센서를 채널로 취급하여 시간축에서 지역 패턴을 학습
- 다운샘플링(60 timesteps)으로 정보 손실 가능 → 다음 노트북에서 multi-rate fusion 시도
- 각 타깃을 독립 학습하므로 타깃 간 상관관계를 활용하지 못함

### Next: `05_multitask.ipynb`
- 공유 backbone + 4개 classification head (Multi-task Learning)
- 타깃 간 상관관계를 활용하여 성능 향상 기대 (H4 가설 검증)
- 학습된 multi-rate alignment 시도 (H2 가설)